# Notizen
## Relevante Themen sind:
- Privatkunden: Wie tief soll es gehen?->auf die Stufe einschränken 5.5 min craw
- Geschäftskunden
- Netze
## Wichtige Frage:
- Nachfragen, wo der Fokus für jedes Thema liegt: für Menü rechts oben, muss man auch alle Untertiteln durchgehen?

# Teil 1: Crawler

In [93]:
from agents import Agent, Tool, Runner, trace, Tool, RunConfig
from contextlib import AsyncExitStack
from agents.mcp import MCPServerStdio
from agents.models.openai_provider import OpenAIProvider
import openai
from rich.markdown import Markdown
from rich.console import Console
import asyncio
from dotenv import load_dotenv
import os

# for asyncio in jupyter notebook
import nest_asyncio
nest_asyncio.apply()

load_dotenv(override=True)

playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"], "client_timeout": 30}
web_scraping_mcp_params = [playwright_params]

### System Prompt for Crawling

In [ ]:
Format = "Markdown-Format"
special_crawling_instructions = f"""
Du bist der Web-Scraping-Agent. Deine Aufgabe ist es, alle Unterthemen einer Webseite systematisch durchzugehen und gezielt FAQ-Inhalte sowie FAQ-relevante Inhalte vollständig zu extrahieren.

Allgemeine Anweisungen:
- Prüfe beim Öffnen einer Webseite immer auf Cookie-Consent-Banner.
- Falls ein Button wie "Alle akzeptieren", "Akzeptieren" oder "Accept all" sichtbar ist, klicke diesen zuerst, bevor du irgendetwas anderes tust.
- Interagiere mit Navigationselementen erst, nachdem alle Overlays geschlossen wurden.

Navigation:
- Falls im User-Prompt bestimmte Themen oder Unterthemen angegeben sind, navigiere ausschließlich zu diesen – halte dich exakt an die vorgegebene Liste.
- Falls im User-Prompt "alles" oder keine spezifische Liste angegeben ist, gehe systematisch durch ALLE Unterthemen und Abschnitte, die auf der Webseite vorhanden sind.
- Folge dabei der Struktur der Webseite und überspringe keinen Bereich.

Inhaltserfassung – NUR FAQ-Bereiche und FAQ-relevante Inhalte:
- Ignoriere alle anderen Inhalte (Einleitungstexte, Werbung, allgemeine Beschreibungen).
- Fokussiere dich ausschließlich auf FAQ-Abschnitte und alle Abschnitte mit Frage-Antwort-Struktur (z.B. 'Sie haben Fragen?', 'Häufige Fragen', aufklappbare Fragen etc.).
- Erfasse JEDES Frage-Antwort-Paar vollständig und wörtlich – genau so, wie es auf der Webseite steht.
- Kürze, paraphrasiere oder verändere keine Fragen oder Antworten.
- Kein Frage-Antwort-Paar darf fehlen.

Anweisung für angehängte Dokumente (PDF / Formulare):
- Falls an eine Frage oder Antwort ein PDF oder Formular angehängt ist, schreibe NICHT den Inhalt heraus.
- Vermerke stattdessen: "📎 [Typ: PDF/Formular] – [Kurzbeschreibung worum es geht] – Zu finden unter: [Abschnitt/Link/Position auf der Seite]"

Ausgabeformat:
- Erstelle eine vollständige Dokumentation im {Format}.
- Behalte die Struktur der Webseite bei – verwende die Themen und Unterthemen der Seite als Gliederung.
- Jedes Frage-Antwort-Paar wird exakt so wiedergegeben:
  **F: [Frage]**
  A: [Antwort]
- Kein FAQ-Paar darf fehlen.
"""

general_crawling_instructions = f"""
Bitte analysiere die folgende Webseite vollständig: {{url}}

Vorgehensweise:
1. Öffne die Webseite und schließe alle Cookie-Banner.
2. Lies den gesamten Inhalt der Seite vollständig und wörtlich durch.
3. Falls ein Abschnitt oder Unterthema klickbar ist, klicke genau einmal darauf und fasse zusammen, worum es geht und wo es zu finden ist.
4. Falls auf dieser Unterseite weitere klickbare Links existieren, klicke diese NICHT an – notiere sie nur mit 🔗.
5. Achte besonders auf FAQ-Bereiche und alle Abschnitte mit Frage-Antwort-Struktur – erfasse diese vollständig und wörtlich.
6. Falls ein PDF oder Formular vorhanden ist, notiere: Typ, worum es geht, und wo es zu finden ist – schreibe den Inhalt NICHT heraus.

Ausgabe:
Erstelle eine strukturierte Dokumentation in {Format} anhand der Seitenstruktur:

# [Hauptabschnitt]
## [Unterabschnitt]
[Vollständiger Inhalt]

### FAQ / Frage-Antwort
**F: [Frage]**
A: [Antwort]

🔗 [Titel] – Zu finden unter: [Position/Link]
📎 [Typ: PDF/Formular] – [Kurzbeschreibung] – Zu finden unter: [Position/Link]
"""

### User Prompt for Crawling

- URLs

In [ ]:
extract_after_underscore = lambda s: s.split("_", 1)[1].capitalize()
url_privatkunden = "https://www.stadtwerke-waiblingen.de/Privatkunden/Strom"
url_geschaeftskunden = "https://www.stadtwerke-waiblingen.de/Geschäftskunden/Strom"
url_netz = "https://www.stadtwerke-waiblingen.de/Netze/Uebersicht-Netze"
url_unternehmen = "https://www.stadtwerke-waiblingen.de/unternehmen"
# frage nach, was hier der individuelle Fokus ist, wenn subtitles  
url_karriere = "https://www.stadtwerke-waiblingen.de/karriere" # Häufig gestellte Fragen
url_aktuelles = "https://www.stadtwerke-waiblingen.de/aktuelles" # akutelle Baustellen
url_stoerung = "https://www.stadtwerke-waiblingen.de/notfallnummern" # Allgemeine Hinweise zum Stromausfall
url_kontakt = "https://www.stadtwerke-waiblingen.de/kontakt"
url_kundenportal = "https://www.stadtwerke-waiblingen.de/kontakt" # info, was das Kundenportal bietet

- Privatkunden

In [162]:
def set_message_privatkunden(url_privatkunden):
    return f"""
    Bitte analysiere die folgende Webseite: {url_privatkunden}

    Navigiere zu den folgenden Themen und Unterthemen und extrahiere ausschließlich die FAQ-Inhalte und die FAQ-relevanten-Inhalte (wie z.B. 'Sie haben Fragen?') aus jedem Bereich:
    -Privatkunden
    - Sie haben Fragen?
    - Strom
      - Preisinformation
        - Erklärungen zu den Umlagen und Abgaben 
      - Sie haben Fragen?
    - Erdgas
      - Zur Grundversorgung
      - Hinweis zur Gasabrechnung
      - Sie haben Fragen?
    - Wasser
      - Sie haben Fragen? 
    - Wärme
      - Fernwärme
        - Schritt für Schritt zum Fernwärmeanschluss
        - Alle Infos zur Versorgung mit Fernwärme
        - Vorteile auf einen Blick
      - Mobile Heizzentralen mieten
        - MHZ Anhänger bis 200 kW
        - MHZ Elektroheizung bis 40 kW
        - MHZ Container bis 455 kW
    - E-Mobilität
      - Anmeldung E-Ladestation
      - Autostromtarif
    - Bäder
      - Tarifübersicht Waiblinger Bäder
    - Service
      - Zählerstand mitteilen
        - Wie lese ich meinen Zählerstand richtig ab?
      - Umzugsservice
        - FAQ Umzugsmeldungen
      - Abrechnung & Zahlung
        - Hinweis zur Gasabrechnung
      - Abschläge berechnen & verstehen
      - Energiesparen
        - Energiespartipps für Zuhause Wenig Aufwand-viel Ersparnis
      - Warnung vor Betrugsversuchen
      - Unsere Öffnungszeiten

    Vorgehensweise:
    1. Öffne die Webseite und schließe alle Cookie-Banner.
    2. Navigiere nacheinander zu jedem oben genannten Thema und Unterthema.
    3. Suche in jedem Bereich gezielt nach dem FAQ-Abschnitt und dem FAQ-relevanten-Abschnitt.
    4. Erfasse jedes Frage-Antwort-Paar vollständig und wörtlich – genau so wie auf der Seite.
    5. Falls ein PDF oder Formular angehängt ist, notiere: Typ, worum es geht, und wo es zu finden ist – aber schreibe den Inhalt NICHT heraus.
    6. Ignoriere alle anderen Inhalte außerhalb der FAQ-Bereiche und FAQ-relevanten-Bereiche.

    Ausgabe:
    Erstelle eine strukturierte Markdown-Dokumentation mit folgender Gliederung:

    # [Thema]
    ## [Unterthema]
    ### FAQ
    **F: [Frage]**
    A: [Antwort]

    📎 [Typ: PDF/Formular] – [Kurzbeschreibung] – Zu finden unter: [Position/Link]
    """


- Geschaeftskunden

In [123]:
def set_message_geschaeftskunden(url_geschaeftskunden):
    return f"""
    Bitte analysiere die folgende Webseite: {url_geschaeftskunden}

    Navigiere zu den folgenden Themen und Unterthemen und extrahiere ausschließlich die FAQ-Inhalte und die FAQ-relevanten-Inhalte (wie z.B. 'Sie haben Fragen?') aus jedem Bereich:

    - Strom
      - Grundversorgung
      - Preisinformation
      - Stromkennzeichnung
    - Erdgas
      - Grundversorgung
    - Wasser
    - Wärme
      - Fernwärme
      - Mobile Heizzentralen mieten
    - Dienstleistungen
      - Energielieferung Strom und Gas
      - E-Mobilität – bei der NEW
      - Photovoltaik-Anlagen
      - Glasfaseranschluss
    - Service
      - Zählerstand mitteilen
      - Kundenportal
      - Abrechnung & Zahlung

    Vorgehensweise:
    1. Öffne die Webseite und schließe alle Cookie-Banner.
    2. Navigiere nacheinander zu jedem oben genannten Thema und Unterthema.
    3. Suche in jedem Bereich gezielt nach dem FAQ-Abschnitt und dem FAQ-relevanten-Abschnitt.
    4. Erfasse jedes Frage-Antwort-Paar vollständig und wörtlich – genau so wie auf der Seite.
    5. Falls ein PDF oder Formular angehängt ist, notiere: Typ, worum es geht, und wo es zu finden ist – aber schreibe den Inhalt NICHT heraus.
    6. Ignoriere alle anderen Inhalte außerhalb der FAQ-Bereiche und FAQ-relevanten-Bereiche.

    Ausgabe:
    Erstelle eine strukturierte Markdown-Dokumentation mit folgender Gliederung:

    # [Thema]
    ## [Unterthema]
    ### FAQ
    **F: [Frage]**
    A: [Antwort]

    📎 [Typ: PDF/Formular] – [Kurzbeschreibung] – Zu finden unter: [Position/Link]
    """

- Netz

In [144]:
def set_message_netz(url_netz):
    return f"""
    Bitte analysiere die folgende Webseite: {url_netz}

    Navigiere zu den folgenden Themen und Unterthemen und extrahiere ausschließlich die FAQ-Inhalte und die FAQ-relevanten-Inhalte (wie z.B. 'Sie haben Fragen?') aus jedem Bereich:

    - Übersicht Netze
    - Strom
      - Anmeldung E-Ladestation
      - Netzanschluss
      - Netzzugang & Verträge
      - Netznutzungsentgelte
      - Netzstrukturdaten
      - Grund- & Ersatzversorgung
      - EEG & Einspeiser
    - Erdgas
      - Netzanschluss
      - Netzzugang & Verträge
      - Netznutzungsentgelte
      - Netzstrukturdaten
      - Grund- & Ersatzversorgung
    - Wasser
    - Glasfaser
    - Messstellenbetrieb
    - Planauskunft
      - Zur Planauskunft

    Vorgehensweise:
    1. Öffne die Webseite und schließe alle Cookie-Banner.
    2. Navigiere nacheinander zu jedem oben genannten Thema und Unterthema.
    3. Suche in jedem Bereich gezielt nach dem FAQ-Abschnitt und dem FAQ-relevanten-Abschnitt.
    4. Erfasse jedes Frage-Antwort-Paar vollständig und wörtlich – genau so wie auf der Seite.
    5. Achte besonders auf folgende Unterthemen – hier existiert ein Formular auf der Webseite:
      - Strom → Anmeldung E-Ladestation
      - Planauskunft → Zur Planauskunft
      Vermerke diese jeweils als: 📎 [Typ: Formular] – [Kurzbeschreibung worum es geht] – Zu finden unter: [Position/Link]
    6. Falls weitere PDFs oder Formulare angehängt sind, notiere ebenfalls: Typ, worum es geht, und wo es zu finden ist – aber schreibe den Inhalt NICHT heraus.
    7. Ignoriere alle anderen Inhalte außerhalb der FAQ-Bereiche und FAQ-relevanten-Bereiche.

    Ausgabe:
    Erstelle eine strukturierte Markdown-Dokumentation mit folgender Gliederung:

    # [Thema]
    ## [Unterthema]
    ### FAQ
    **F: [Frage]**
    A: [Antwort]

    📎 [Typ: PDF/Formular] – [Kurzbeschreibung] – Zu finden unter: [Position/Link]
    """

- General subtopics

In [145]:
def set_message_general(url):
    return f"""
    Bitte analysiere die folgende Webseite vollständig: {url}

    Vorgehensweise:
    1. Öffne die Webseite und schließe alle Cookie-Banner.
    2. Lies den gesamten Inhalt der Seite vollständig und wörtlich durch.
    3. Falls ein Abschnitt oder Unterthema klickbar ist, klicke genau einmal darauf und fasse zusammen, worum es geht und wo es zu finden ist.
    4. Falls auf dieser Unterseite weitere klickbare Links existieren, klicke diese NICHT an – notiere sie nur mit 🔗.
    5. Achte besonders auf FAQ-Bereiche und alle Abschnitte mit Frage-Antwort-Struktur – erfasse diese vollständig und wörtlich.
    6. Falls ein PDF oder Formular vorhanden ist, notiere: Typ, worum es geht, und wo es zu finden ist – schreibe den Inhalt NICHT heraus.

    Ausgabe:
    Erstelle eine strukturierte Markdown-Dokumentation anhand der Seitenstruktur:

    # [Hauptabschnitt]
    ## [Unterabschnitt]
    [Vollständiger Inhalt]

    ### FAQ / Frage-Antwort
    **F: [Frage]**
    A: [Antwort]

    🔗 [Titel] – Zu finden unter: [Position/Link]
    📎 [Typ: PDF/Formular] – [Kurzbeschreibung] – Zu finden unter: [Position/Link]
    """

In [146]:
configs = {
    "privatkunden": {
        "url": url_privatkunden,
        "system_prompt": special_crawling_instructions,
        "prompt": set_message_privatkunden,
    },
    "geschaeftskunden": {
        "url": url_geschaeftskunden,
        "system_prompt": special_crawling_instructions,
        "prompt": set_message_geschaeftskunden,
    },
    "netz": {
        "url": url_netz,
        "system_prompt": special_crawling_instructions,
        "prompt": set_message_netz,
    },
    "general": {
        "url": "general",
        "system_prompt": set_message_general,
        "prompt": general_crawling_instructions,
    },    
}

In [147]:
configs["netz"]["url"]

'https://www.stadtwerke-waiblingen.de/Netze/Uebersicht-Netze'

In [160]:
from pathlib import Path
from datetime import datetime

def save_markdown(markdown_text: str,output_dir: str = "outputs",filename: str | None = None,encoding: str = "utf-8") -> Path:
    """
    Speichert Markdown-Text lokal in einer .md-Datei.

    Args:
        markdown_text: Der Markdown-Inhalt (String)
        output_dir: Zielordner (wird erstellt, falls nicht vorhanden)
        filename: Dateiname ohne Endung; falls None → Zeitstempel
        encoding: Text-Encoding (Standard: utf-8)

    Returns:
        Path zur gespeicherten Datei
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    #if filename is None:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{filename}_{timestamp}"

    file_path = output_path / f"{filename}.md"

    with file_path.open("w", encoding=encoding) as f:
        f.write(markdown_text.strip() + "\n")

    return file_path

## Set topic

In [149]:
urls = {
    "privatkunden": url_privatkunden,
    "geschaeftskunden": url_geschaeftskunden,
    "netz": url_netz,
    "unternehmen": url_unternehmen,
    "karriere": url_karriere,
    "aktuelles": url_aktuelles,
    "stoerung": url_stoerung,
    "kontakt": url_kontakt,
    "kundenportal": url_kundenportal
}

In [156]:
topic = "privatkunden"
if topic in configs.keys():
    instructions = configs[topic]["system_prompt"]
    url = urls[topic]
    message = configs[topic]["prompt"](url)
else:
    instructions = general_crawling_instructions
    url = urls[topic]
    message = set_message_general(url)
#for 3 basics kategory should be the same as others

In [157]:
message

"\n    Bitte analysiere die folgende Webseite: https://www.stadtwerke-waiblingen.de/Privatkunden/Strom\n\n    Navigiere zu den folgenden Themen und Unterthemen und extrahiere ausschließlich die FAQ-Inhalte und die FAQ-relevanten-Inhalte (wie z.B. 'Sie haben Fragen?') aus jedem Bereich:\n\n    - Strom\n      - Ökostrom\n      - Wärmestrom\n      - Grundversorgung\n      - Preisinformation\n      - Stromkennzeichnung\n    - Erdgas\n      - Grundversorgung\n    - Wasser\n    - Wärme\n      - Fernwärme\n      - Mobile Heizzentralen mieten\n    - E-Mobilität\n      - Anmeldung E-Ladestation\n    - Bäder\n    - Service\n      - Zählerstand mitteilen\n      - Umzugsservice\n      - Abrechnung & Zahlung\n      - Abschläge berechnen & verstehen\n      - Energiesparen\n      - Warnung vor Betrugsversuchen\n\n    Vorgehensweise:\n    1. Öffne die Webseite und schließe alle Cookie-Banner.\n    2. Navigiere nacheinander zu jedem oben genannten Thema und Unterthema.\n    3. Suche in jedem Bereich ge

## Do crawling

In [158]:
# Create a client with built-in retry logic
openai_client = openai.AsyncOpenAI(
    max_retries=5,  # retry up to 5 times
)

run_config = RunConfig(
    model_provider=OpenAIProvider(openai_client=openai_client)
)

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=120) as craw_server:
    agent = Agent(
        name="agent", 
        instructions=instructions, 
        model="gpt-4.1-mini", 
        mcp_servers=[craw_server]
    )
    with trace(f"{topic}"):
        result = await Runner.run(agent, message, max_turns=100, run_config=run_config)

display(Markdown(result.final_output))

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                      Strom                                                      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛


                                                     Ökostrom                                                      

                                                        FAQ                                                        

F: Kunden-Center                                                                                                   
A: +49 7151 131-170                                                                                                


                                                    Wärmestrom                                                     

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    


                                                  Grundversorgung                                                  

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    


                                                 Preisinformation                                                  

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    


                                                Stromkennzeichnung                                                 

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                     Erdgas                                                      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛


                                                  Grundversorgung                                                  

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                     Wasser                                                      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                                                        FAQ                                                        

Keine FAQ oder FAQ-relevanten Inhalte gefunden.                                                                    

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                      Wärme                                                      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛


                                                     Fernwärme                                                     

                        

## Save as Markdown

In [161]:
save_markdown(
    markdown_text=result.final_output, 
    output_dir="/Users/sunzeyuan/projects/agent_exercises/agents/crawler/output", 
    filename = f"{topic}_Zusammenfassung"
)

PosixPath('/Users/sunzeyuan/projects/agent_exercises/agents/crawler/output/privatkunden_Zusammenfassung_20260303_215547.md')

In [ ]:
import asyncio

tasks = []
for name, msg in conversations.items():
    tasks.append(
        asyncio.create_task(
            run_single_conversation(name, msg)
        )
    )

results = await asyncio.gather(*tasks)

In [ ]:
async def run_single_conversation(name, message):
    with trace(f"{name}"):
        return await Runner.run(agent, message, max_turns=100, run_config=run_config)

# Databank - Knowledge-graph based

In [ ]:
from contextlib import AsyncExitStack
from agents import Agent, Runner, trace, Tool
from IPython.display import Markdown
import asyncio
from rich.markdown import Markdown
from rich.console import Console

from agents.mcp import MCPServerStdio

#from templates import tex_writer_instructions, evaluator_instructions
#from databank_management import get_databank_management_agent_tool
#from tools import tex_to_pdf
#from mcp_params import files_params, memory_params

In [ ]:
databank_management_instructions =  """
You are an agent responsible for managing a knowledge-graph-based databank.

Your tasks include:
• Saving information: Whenever asked to save information, store all details that could be useful in the future. Be thorough, structured, and comprehensive. Do not omit details unless explicitly told to.
• Retrieving information: When asked to retrieve information, provide a well-structured, clearly written summary containing all details for the requested topic.

Only perform databank-related actions (saving or retrieving). For anything unclear, or if you need more information, ask the user directly.
"""

In [ ]:
async def get_databank_management_agent(databank_management_mcp_servers) -> Agent:
    databank_management_agent = Agent(
        name="databank_management_agent",
        instructions=databank_management_instructions,
        mcp_servers=databank_management_mcp_servers,
        model="gpt-4o-mini",
    )
    return databank_management_agent

async def get_databank_management_agent_tool(databank_management_mcp_servers) -> Tool:
    databank_management_agent = await get_databank_management_agent(databank_management_mcp_servers)
    return databank_management_agent.as_tool(
            tool_name="databank_management_agent_tool",
            tool_description=(
                "Use this tool to save or retrieve information from the knowledge-graph databank." 
                "Always provide clear and comprehensive summaries when retrieving data."
                )

        )

async def run():
    async with AsyncExitStack() as stack:
        
        databank_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in databank_management_mcp_params]

        databank_management_agent = await get_databank_management_agent(databank_servers)

        message = "What information do you have in your databank? Show me a detailed summary."

        with trace("test databank management agent"):
            result = await Runner.run(databank_management_agent, message, max_turns=10)
        #print(result.final_output) # if prefer desplay plain text
        console = Console()
        console.print(Markdown(result.final_output))